# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [17]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [19]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "TODO: replace with a short sentence you want to tokenize"
print(sample_sentence)


TODO: replace with a short sentence you want to tokenize


In [18]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | tod          | 28681
    2 | ##o          |  2080
    3 | :            |  1024
    4 | replace      |  5672
    5 | with         |  2007
    6 | a            |  1037
    7 | short        |  2460
    8 | sentence     |  6251
    9 | you          |  2017
   10 | want         |  2215
   11 | to           |  2000
   12 | token        | 19204
   13 | ##ize        |  4697
   14 | [SEP]        |   102
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (14, '[SEP]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]'), (20, '[PAD]'), (21, '[PAD]

### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [20]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "TODO: add a sentence whose sentiment you want to test"
prediction = sentiment_pipeline(sentence)
prediction


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9761614203453064}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

In [22]:
class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        '''TODO: load the tokenizer/model and move the model to the proper device.'''
        raise NotImplementedError("Initialize tokenizer, model, and device here.")

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        '''TODO: clean the text, tokenize, and return tensors ready for inference.'''
        raise NotImplementedError("Return a dict of tensors produced by the tokenizer.")

    def predict(self, text: str) -> Dict[str, float]:
        '''TODO: run a forward pass, apply softmax, and return a label plus probability.'''
        raise NotImplementedError("Add inference and post-processing logic.")


In [24]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
# analyzer = BERTSentimentAnalyzer()
# samples = [
#     "TODO: add a clearly positive statement",
#     "TODO: add a clearly negative statement"
# ]
# for text in samples:
#     print(text)
#     print(analyzer.predict(text))


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [26]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

In [28]:
class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        '''TODO: load the tokenizer and model, and detect the available device.'''
        raise NotImplementedError("Initialize tokenizer, model, and device.")

    def recognize(self, text: str):
        '''TODO: tokenize the text, run the model, map predictions to BIO labels, and merge word pieces.'''
        raise NotImplementedError("Return structured entities.")


In [27]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
# ner = BERTNamedEntityRecognizer()
# sample_text = "TODO: add a short paragraph with at least two entities."
# ner.recognize(sample_text)


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | TODO | TODO |
| Primary purpose | TODO | TODO |
| Typical use cases | TODO | TODO |
| Strengths | TODO | TODO |
| Weaknesses | TODO | TODO |


In [30]:
class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        '''TODO: load the tokenizer and model, and detect the available device.'''
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def recognize(self, text: str):
        '''TODO: tokenize the text, run the model, map predictions to BIO labels, and merge word pieces.'''
        raise NotImplementedError("Return structured entities.")

## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.
2. TODO: Explain how those embeddings are stored and searched in a vector database.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.


How BERT encodes queries and documents BERT encode les requêtes et les documents en les transformant en vecteurs denses de haute dimension (typiquement 768 dimensions pour BERT-base) qui capturent leur sens sémantique profond. Pour une requête (ex. : la question d'un utilisateur), le texte est tokenisé, préfixé de [CLS], puis passé dans les couches du transformer. L'embedding final du token [CLS] en sortie du dernier encodeur — ou une moyenne des embeddings de tous les tokens (mean pooling) — devient le vecteur représentatif de la requête. Le même processus s'applique à chaque document (ou passage) de la base de connaissances : chaque texte produit un vecteur dense qui encode son contenu sémantique. Dans les pipelines RAG modernes, on utilise souvent des variantes spécialisées comme Sentence-BERT (SBERT), entraîné avec des paires de phrases similaires/dissimilaires pour produire des embeddings particulièrement adaptés à la comparaison sémantique par similarité cosinus.

How those embeddings are stored and searched in a vector database Une fois calculés, les vecteurs des documents sont indexés dans une base de données vectorielle (ex. : FAISS, Pinecone, Weaviate, Chroma). Ces systèmes sont optimisés pour la recherche par plus proche voisin approximatif (ANN — Approximate Nearest Neighbor), permettant de parcourir des millions de vecteurs en quelques millisecondes. Lors d'une requête, le vecteur de la question est comparé aux vecteurs stockés via la similarité cosinus ou le produit scalaire : sim(q,d)=q⋅d∥q∥⋅∥d∥\text{sim}(q, d) = \frac{q \cdot d}{|q| \cdot |d|}sim(q,d)=∥q∥⋅∥d∥q⋅d​Les k documents ayant les scores de similarité les plus élevés sont retournés comme passages pertinents. L'indexation utilise des structures comme les graphes HNSW (Hierarchical Navigable Small World) ou les index IVF (Inverted File) pour éviter une recherche exhaustive et garantir des temps de réponse compatibles avec une utilisation en production.

How the retrieved passages are handed to a generative model like GPT Les passages récupérés sont concaténés avec la question originale pour former un prompt enrichi, qui est ensuite transmis au modèle génératif. La structure typique du prompt est : Contexte : [Passage 1 récupéré] [Passage 2 récupéré] [Passage 3 récupéré]

Question : <question de l'utilisateur>

Réponse : Le modèle génératif (GPT-4, LLaMA, Mistral…) n'a pas besoin de mémoriser toutes les connaissances dans ses poids — il raisonne à partir du contexte fourni. Cela résout deux problèmes majeurs des LLMs purs : les hallucinations (le modèle invente des faits) et le knowledge cutoff (les connaissances figées à la date d'entraînement). Le modèle génère alors une réponse ancrée dans des sources réelles et vérifiables, ce qui améliore la fiabilité et la traçabilité des sorties.

Concrete application example Cas concret : Support client intelligent dans le secteur bancaire Une banque comme Société Générale ou Ecobank gère des milliers de documents internes — contrats, fiches produits, réglementations, procédures de conformité. Sans RAG, un employé du service client doit chercher manuellement dans ces documents pour répondre à un client. Avec un pipeline RAG + BERT :
Tous les documents internes sont encodés par SBERT et indexés dans une base vectorielle (ex. : Pinecone). Quand un agent tape : "Quelles sont les conditions de résiliation anticipée d'un prêt immobilier ?", son embedding est calculé et les 3 passages les plus pertinents du contrat de prêt sont récupérés en <100ms. Ces passages sont injectés dans le prompt d'un LLM (GPT-4) qui génère une réponse précise, citant les clauses exactes.

Avantages dans ce contexte :

La base documentaire peut être mise à jour (nouveau règlement, nouveau produit) sans réentraîner le LLM — il suffit de ré-indexer les nouveaux documents. Les réponses sont traçables : on peut afficher les sources au conseiller pour qu'il vérifie. Fonctionne en multilingue avec des modèles BERT multilingues (mBERT, XLM-R), utile pour des banques opérant en Afrique de l'Ouest avec plusieurs langues.
